# Baseline feature extraction

Notebook ini mengekstrak fitur operasional gegenpressing untuk seluruh queue
anotasi. Fitur hanya berasal dari tracking pada dua detik setelah loss. Outcome
quick regain dan label manual tidak digunakan selama ekstraksi, sehingga tidak
terjadi target leakage.

Metodologi tidak berubah: fitur mencakup jarak, approach rate, jumlah presser,
compactness, dan durasi tekanan. Tabel ini disiapkan untuk baseline setelah
anotasi penuh selesai.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.gegenpressing_features import (
    BASELINE_FEATURE_COLUMNS,
    audit_baseline_feature_table,
    build_baseline_feature_table,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'sportec_idsse'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ANNOTATION_PATH = PROCESSED_DIR / 'gegenpressing_annotation_manifest.csv'
FEATURE_PATH = PROCESSED_DIR / 'gegenpressing_baseline_features.csv'

In [2]:
annotations = pd.read_csv(ANNOTATION_PATH)
features = build_baseline_feature_table(annotations, DATA_DIR)
features.to_csv(FEATURE_PATH, index=False)

print('Candidates:', len(annotations))
print('Feature rows:', len(features))
print('Feature columns:', len(BASELINE_FEATURE_COLUMNS))
print('Saved:', FEATURE_PATH)

Candidates: 1873
Feature rows: 1873
Feature columns: 12
Saved: c:\VAEP-Track\data\processed\gegenpressing_baseline_features.csv


In [3]:
checks = audit_baseline_feature_table(features, annotations)
display(checks.to_frame())
assert checks.all(), 'Audit baseline feature table gagal.'

missing = features[BASELINE_FEATURE_COLUMNS].isna().sum()
print('Missing values by feature:')
display(missing[missing.gt(0)].rename('missing').to_frame())
print('Feature summary:')
display(features[BASELINE_FEATURE_COLUMNS].describe().T)

,passed
candidate_id_unique,True
candidate_ids_complete,True
feature_columns_complete,True
no_infinite_numeric_values,True
outcome_not_in_features,True
gegenpressing_label_not_in_features,True


Missing values by feature:


,missing
mean_local_compactness_m,88


Feature summary:


,count,mean,std,min,25%,50%,75%,max
max_total_pressers,1873.0,2.534437,1.293068,0.000000,2.000000,2.000000,3.000000,10.000000
max_direct_pressers,1873.0,1.557395,0.908789,0.000000,1.000000,1.000000,2.000000,9.000000
max_approaching_pressers,1873.0,2.041111,1.127510,0.000000,1.000000,2.000000,3.000000,9.000000
pressure_duration_s,1873.0,1.559338,0.640752,0.000000,1.080000,2.040000,2.040000,2.040000
collective_pressure_duration_s,1873.0,0.738708,0.676993,0.000000,0.120000,0.560000,1.200000,2.040000
minimum_losing_distance_m,1873.0,1.686109,2.649870,0.000000,0.422019,0.930376,1.800805,27.562765
maximum_approach_rate_mps,1873.0,12.823447,7.349879,-3.664071,8.103800,11.800984,16.345024,124.667613
max_losing_players_within_10m,1873.0,3.231714,1.606849,0.000000,2.000000,3.000000,4.000000,11.000000
mean_local_compactness_m,1785.0,6.792138,1.465682,1.867653,5.841439,6.876026,7.804851,12.577848
team_stretch_change_m,1873.0,-0.288050,1.121742,-4.709183,-0.979996,-0.242193,0.445367,3.624808


In [4]:
labels = annotations[[
    'candidate_id',
    'is_gegenpressing',
    'annotation_confidence',
]].copy()
audit = features.merge(labels, on='candidate_id', how='left', validate='one_to_one')
completed = audit[audit['is_gegenpressing'].notna()].copy()
usable = completed[completed['is_gegenpressing'].isin([0, 1])].copy()

print('Completed annotations:', len(completed))
print('Usable binary labels:', len(usable))
print('Remaining annotations:', len(audit) - len(completed))
if len(usable):
    display(
        usable['is_gegenpressing']
        .astype(int)
        .value_counts()
        .sort_index()
        .rename('count')
        .to_frame()
    )
print('Status: features ready; final baseline waits for full annotation.')

Completed annotations: 1873
Usable binary labels: 1873
Remaining annotations: 0


,count
is_gegenpressing,
0,292
1,1581


Status: features ready; final baseline waits for full annotation.
